# AI-Powered HR Assistant using RAG

## Overview
This notebook implements an AI-powered HR Assistant that uses Retrieval-Augmented Generation (RAG) to answer questions about Nestlé's HR policies. The system loads the HR policy PDF, creates vector embeddings, and uses an LLM to provide accurate, context-aware responses.

### Features:
- **Document Loading**: Extracts content from Nestlé HR policy PDF
- **Vector Embeddings**: Converts text into embeddings using OpenAI API
- **Vector Database**: Stores embeddings in FAISS for efficient retrieval
- **RAG Pipeline**: Combines document retrieval with LLM for intelligent responses
- **Interactive Interface**: Simple chatbot for HR policy queries

## 1. Import Required Libraries

In [6]:
# Install required packages
import subprocess
import sys

packages = [
    'langchain',
    'openai',
    'pypdf',
    'faiss-cpu',
    'numpy',
    'pandas',
    'python-dotenv'
]

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

# Import all required libraries
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any



✓ langchain already installed
✓ openai already installed
✓ pypdf already installed
Installing faiss-cpu...
✓ faiss-cpu installed
✓ numpy already installed
✓ pandas already installed
Installing python-dotenv...
✓ python-dotenv installed


In [9]:
# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# PDF processing
from pypdf import PdfReader

print("\n✓ All libraries imported successfully!")


✓ All libraries imported successfully!


In [10]:
# Configure OpenAI API
# You need to set your OpenAI API key here
# Option 1: Set it directly (for testing only - not recommended for production)
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

# Option 2: Use .env file or environment variable
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("\n⚠️  WARNING: OPENAI_API_KEY environment variable not set!")
    print("Please set your OpenAI API key to use this notebook.")
    print("\nTo set it:")
    print("  Using .env file: Create a .env file with OPENAI_API_KEY=your-key")
    print("  Or set environment variable: export OPENAI_API_KEY='your-key'")
    print("\nYou can get your API key from: https://platform.openai.com/api-keys")
else:
    print("✓ OpenAI API Key configured successfully!")

# Define paths
dataset_path = Path("./Dataset/the_nestle_hr_policy_pdf_2012.pdf")
print(f"\nDataset path: {dataset_path}")
print(f"Dataset exists: {dataset_path.exists()}")

✓ OpenAI API Key configured successfully!

Dataset path: Dataset\the_nestle_hr_policy_pdf_2012.pdf
Dataset exists: True


## 2. Load and Parse HR Policy PDF

In [11]:
def load_pdf_content(pdf_path: str) -> str:
    """
    Load and extract text content from a PDF file.
    
    Args:
        pdf_path: Path to the PDF file
        
    Returns:
        Extracted text content from the PDF
    """
    try:
        pdf_reader = PdfReader(pdf_path)
        extracted_text = ""
        
        print(f"Loading PDF from: {pdf_path}")
        print(f"Total pages: {len(pdf_reader.pages)}")
        
        for page_num, page in enumerate(pdf_reader.pages):
            text = page.extract_text()
            extracted_text += text + "\n"
            
            if (page_num + 1) % 10 == 0:
                print(f"Processed {page_num + 1} pages...")
        
        print(f"✓ Successfully extracted text from {len(pdf_reader.pages)} pages")
        print(f"Total characters: {len(extracted_text)}")
        
        return extracted_text
    
    except Exception as e:
        print(f"Error loading PDF: {str(e)}")
        raise

# Load the HR policy document
if dataset_path.exists():
    raw_documents = load_pdf_content(str(dataset_path))
    print(f"\n✓ PDF loaded successfully!")
    print(f"First 500 characters:\n{raw_documents[:500]}\n...")
else:
    print(f"❌ PDF file not found at {dataset_path}")

Loading PDF from: Dataset\the_nestle_hr_policy_pdf_2012.pdf
Total pages: 8
✓ Successfully extracted text from 8 pages
Total characters: 14331

✓ PDF loaded successfully!
First 500 characters:
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy
Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be found in the Centre online repository at:  
http://intranet.nestle.com/nestledocs
Copyright
 and  confidentiality
Al
l rights belong to Nestec Ltd., Vevey, Switzerland.
© 2012, Nestec Ltd.
Design
Nestec 
...


In [18]:
def split_text_into_chunks(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    
    chunks = text_splitter.split_text(text)
    print(f"Text split into {len(chunks)} chunks")
    
    return chunks

# Split the documents into chunks
document_chunks = split_text_into_chunks(raw_documents, chunk_size=1000, chunk_overlap=200)

# Show sample chunks
print(f"\nSample chunks (first 3):")
print("=" * 80)
for i, chunk in enumerate(document_chunks[:3]):
    print(f"\nChunk {i+1}:")
    print(chunk[:300] + "..." if len(chunk) > 300 else chunk)
    print("=" * 80)

Text split into 18 chunks

Sample chunks (first 3):

Chunk 1:
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy
Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Board, Nestlé S.A.
Repository
All Nestlé Principles and Policies, Standards and  
Guidelines can be fou...

Chunk 2:
1
At Nestlé, we recognize that our employees 
are the key to our success and nothing can be 
achieved without their engagement. 
This document encompasses the guidelines 
which constitute a solid basis for effective Human 
Resources Management throughout the Nestlé 
Group around the world. It explai...

Chunk 3:
The implementation of this policy will be 
inspired by sound judgement, compliance with 
local market laws and common sense, taking into 
account the specific context. Its spirit should be 
respected under all circumstances and could be 
summarised in one sentence: At Nestlé we put 
people at the ce...


## 3. Create Vector Embeddings and Store in Vector Database

In [21]:
# Create embeddings and vector store
if openai_api_key:
    try:
        print("Creating OpenAI embeddings...")
        embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small')
        
        # Create FAISS vector store
        vector_store = FAISS.from_texts(
            texts=document_chunks,
            embedding=embeddings
        )

        # Test retrieval
        print("\nTesting retrieval with a sample query...")
        test_query = "What is the leave policy?"
        test_results = vector_store.similarity_search(test_query, k=2)
        
        print(f"Found {len(test_results)} relevant documents for: '{test_query}'")
        print(f"Top result preview: {test_results[0].page_content[:200]}...")
        
    except Exception as e:
        print(f"❌ Error creating embeddings: {str(e)}")
        print("Make sure your OpenAI API key is valid and you have sufficient credits.")
        vector_store = None
else:
    print("❌ Cannot create embeddings without OpenAI API key")
    vector_store = None

Creating OpenAI embeddings...

Testing retrieval with a sample query...
Found 2 relevant documents for: 'What is the leave policy?'
Top result preview: necessary for the continued development of 
people and the Company.
Corporate policy: 
Expatriation Policy
 Talent, development  
 and performance management
The Nestlé Human Resources Policy
5
Since ...


## 4. Build RAG Pipeline with LLM

In [26]:
# Create prompt template for HR Assistant
hr_assistant_prompt = """Use the following pieces of context about Nestlé's HR policies to answer the user's question.
If you don't find relevant information in the provided context, say "I don't have information about this in the HR policy document."

IMPORTANT: 
- Always base your answers on the provided HR policy context
- Be clear and concise in your responses
- If a policy requires further action or consultation, mention that

Context from HR Policy:
{context}

User Question: {question}

HR Assistant Response:"""

PROMPT = PromptTemplate(
    template=hr_assistant_prompt,
    input_variables=["context", "question"]
)

# Create the RAG pipeline
if vector_store and openai_api_key:
    try:
        print("Creating RAG pipeline...")
        
        # Initialize the LLM
        llm = ChatOpenAI(
            openai_api_key=openai_api_key,
            model_name="gpt-4o-mini",
            temperature=0.3
        )
        
        # Create RetrievalQA chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",
            retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
            return_source_documents=True,
            chain_type_kwargs={"prompt": PROMPT}
        )
        
        print("✓ RAG pipeline created successfully!")
        print("\nPipeline Components:")
        print("  - LLM: gpt-4o-mini")
        print("  - Vector Store: FAISS")
        print("  - Retrieval: Similarity-based (top 3 documents)")
        
    except Exception as e:
        print(f"❌ Error creating RAG pipeline: {str(e)}")
        qa_chain = None
else:
    print("❌ Cannot create RAG pipeline without vector store and API key")
    qa_chain = None

Creating RAG pipeline...
✓ RAG pipeline created successfully!

Pipeline Components:
  - LLM: gpt-4o-mini
  - Vector Store: FAISS
  - Retrieval: Similarity-based (top 3 documents)


## 5. Create Chatbot Interface

In [27]:
class HRAssistant:
   
    
    def __init__(self, qa_chain, vector_store):
        
        self.qa_chain = qa_chain
        self.vector_store = vector_store
        self.conversation_history = []
    
    def answer_question(self, question: str) -> Dict[str, Any]:
       
        if not self.qa_chain:
            return {
                "answer": "Error: RAG pipeline not initialized. Please check your OpenAI API key.",
                "sources": [],
                "error": True
            }
        
        try:
            # Get response from QA chain
            response = self.qa_chain({"query": question})
            
            # Extract source documents
            source_docs = response.get("source_documents", [])
            sources = [
                {
                    "content": doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content,
                    "full_content": doc.page_content
                }
                for doc in source_docs
            ]
            
            # Store in conversation history
            self.conversation_history.append({
                "question": question,
                "answer": response["result"],
                "sources": sources
            })
            
            return {
                "answer": response["result"],
                "sources": sources,
                "error": False
            }
        
        except Exception as e:
            return {
                "answer": f"Error processing question: {str(e)}",
                "sources": [],
                "error": True
            }
    
    def get_conversation_history(self) -> List[Dict]:
        """Get the conversation history."""
        return self.conversation_history
    
    def clear_history(self):
        """Clear the conversation history."""
        self.conversation_history = []


# Create HR Assistant instance
if qa_chain and vector_store:
    hr_assistant = HRAssistant(qa_chain, vector_store)
    print("✓ HR Assistant initialized successfully!")
    print("\nYou can now ask questions about Nestlé's HR policies.")
    print("Use: hr_assistant.answer_question('Your question here')")
else:
    hr_assistant = None
    print("❌ Could not initialize HR Assistant. Please check the setup.")

✓ HR Assistant initialized successfully!

You can now ask questions about Nestlé's HR policies.
Use: hr_assistant.answer_question('Your question here')


## 6. Test with Sample HR Queries

In [29]:
# Test the HR Assistant with sample queries
if hr_assistant:
    # Sample HR Policy Questions
    sample_queries = [
        "What is the annual leave policy?",
        "What are the employee benefits?",
        "How is sick leave handled?",
        "What is the maternity policy?",
        "What are the working hours?",
        "How are performance appraisals conducted?"
    ]
    
    print("=" * 80)
    print("HR ASSISTANT - TESTING WITH SAMPLE QUERIES")
    print("=" * 80)
    
    # Test with first 3 queries
    for i, query in enumerate(sample_queries[:3], 1):
        print(f"\n\n{'='*80}")
        print(f"QUERY {i}: {query}")
        print(f"{'='*80}\n")
        
        response = hr_assistant.answer_question(query)
        
        if not response["error"]:
            print(f"ANSWER:\n{response['answer']}\n")
            
            if response["sources"]:
                print(f"\nSOURCE DOCUMENTS ({len(response['sources'])} found):")
                print("-" * 80)
                for j, source in enumerate(response["sources"], 1):
                    print(f"\nSource {j}:")
                    print(source["content"])
        else:
            print(f"ERROR: {response['answer']}")
    
    print(f"\n\n{'='*80}")
    print("CONVERSATION HISTORY SUMMARY")
    print(f"{'='*80}")
    print(f"Total queries processed: {len(hr_assistant.get_conversation_history())}")
    
else:
    print("❌ HR Assistant not available. Please ensure all setup steps completed successfully.")

HR ASSISTANT - TESTING WITH SAMPLE QUERIES


QUERY 1: What is the annual leave policy?

ANSWER:
I don't have information about this in the HR policy document.


SOURCE DOCUMENTS (3 found):
--------------------------------------------------------------------------------

Source 1:
necessary for the continued development of 
people and the Company.
Corporate policy: 
Expatriation Policy
 Talent, development  
 and performance management
The Nestlé Human Resources Policy
5
Since ...

Source 2:
does not replace the close relationship that our 
management maintains with all employees.
In the spirit of continuous improvement, we 
encourage two-way dialogue with our employees 
that goes beyond ...

Source 3:
Policy
Mandatory
September   2012
The Nestlé  
Human Resources Policy
Policy
Mandatory
September 
 20
12
Issuing  departement
Hum
an Resources
T arget  audience  
All
 employees
Approver
Executive Boa...


QUERY 2: What are the employee benefits?

ANSWER:
Nestlé's employee benefits are pa

## 7. Interactive Chatbot Function

In [30]:
def display_response(response: Dict[str, Any], show_sources: bool = True) -> None:
    """
    Display the chatbot response in a formatted way.
    
    Args:
        response: Response dictionary from the HR Assistant
        show_sources: Whether to display source documents
    """
    if response["error"]:
        print(f"❌ {response['answer']}")
    else:
        print("\n" + "="*80)
        print("HR ASSISTANT RESPONSE")
        print("="*80)
        print(f"\n{response['answer']}")
        
        if show_sources and response["sources"]:
            print("\n" + "-"*80)
            print(f"SOURCE DOCUMENTS ({len(response['sources'])} found):")
            print("-"*80)
            for i, source in enumerate(response["sources"], 1):
                print(f"\nSource {i}:")
                print(source["content"])
                print()


def interactive_chat(num_rounds: int = 3) -> None:
    """
    Run an interactive chat session with the HR Assistant.
    
    Args:
        num_rounds: Number of questions to ask
    """
    if not hr_assistant:
        print("❌ HR Assistant not initialized.")
        return
    
    print("\n" + "="*80)
    print("NESTLÉ HR ASSISTANT - INTERACTIVE MODE")
    print("="*80)
    print("\nAsk questions about Nestlé's HR policies.")
    print("Type 'quit' to exit, 'history' to see conversation history.\n")
    
    for round_num in range(num_rounds):
        user_input = input(f"\nRound {round_num + 1} - Your question: ").strip()
        
        if user_input.lower() == 'quit':
            print("\nThank you for using the HR Assistant. Goodbye!")
            break
        
        if user_input.lower() == 'history':
            history = hr_assistant.get_conversation_history()
            if history:
                print(f"\n{'='*80}")
                print("CONVERSATION HISTORY")
                print(f"{'='*80}")
                for i, item in enumerate(history, 1):
                    print(f"\n{i}. Q: {item['question']}")
                    print(f"   A: {item['answer'][:100]}...")
            else:
                print("\nNo conversation history yet.")
            continue
        
        if not user_input:
            print("Please enter a question.")
            continue
        
        print("\nProcessing your question...")
        response = hr_assistant.answer_question(user_input)
        display_response(response)


# Example usage
print("\n" + "="*80)
print("EXAMPLE: HOW TO USE THE HR ASSISTANT")
print("="*80)
print("""
1. To ask a single question:
   response = hr_assistant.answer_question("What is the leave policy?")
   display_response(response)

2. For interactive chat (uncomment below and run):
   interactive_chat(num_rounds=5)

3. To get conversation history:
   history = hr_assistant.get_conversation_history()
   
4. To clear conversation history:
   hr_assistant.clear_history()
""")

print("\n✓ HR Assistant is ready to answer your questions about Nestlé HR policies!")


EXAMPLE: HOW TO USE THE HR ASSISTANT

1. To ask a single question:
   response = hr_assistant.answer_question("What is the leave policy?")
   display_response(response)

2. For interactive chat (uncomment below and run):
   interactive_chat(num_rounds=5)

3. To get conversation history:
   history = hr_assistant.get_conversation_history()

4. To clear conversation history:
   hr_assistant.clear_history()


✓ HR Assistant is ready to answer your questions about Nestlé HR policies!
